# 30d-jenny：对数增长率 × 波动率联合分布

本 notebook 是 GMM/HMM 状态模型的第 0 步：先从日销量构造**带符号对数增长率**与滚动波动率，估计其联合概率密度并绘制 3D 密度图。

- `log_growth_t = log1p(qty_t) - log1p(qty_(t-1))`：正值为增长，负值为下降；不把增长率和下降率拆成两套不连续变量。
- `volatility_t`：截至 t 的过去 7 个工作日对数增长率标准差。
- 为避免跨月首日与上月末混合节奏，增长率和波动率均在 `bizym` 内计算，并复用中国法定工作日口径。
- 本页只做 EDA；后续 GMM/HMM 拟合时，训练/验证切点必须分别重算标准化、KDE/GMM/HMM，不能使用全历史参数。

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

VOL_WINDOW = 7
MIN_PERIODS = 5
GRID_SIZE = 100

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'code' / '30d-jenny').is_dir():
            return candidate
    raise FileNotFoundError('Cannot locate repository root containing code/30d-jenny.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
# Match the reproduction README. Set DATA_PATH explicitly only for a deliberate alternate dataset.
DATA_PATH = REPO_ROOT / 'data' / 'sales_30d_daily.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Cannot locate default input: {DATA_PATH}')

print(f'Using data: {DATA_PATH.resolve()}')

Using data: /Users/mark/Git/lab/ts-forecast/data/sales_30d_daily.csv


In [2]:
required_cols = {'bizym', 'transdate', 'num_hosp', 'qty'}
daily_raw = pd.read_csv(DATA_PATH)
missing_cols = required_cols.difference(daily_raw.columns)
if missing_cols:
    raise ValueError(f'Missing required columns: {sorted(missing_cols)}')

daily_observed = (
    daily_raw.assign(transdate=lambda frame: pd.to_datetime(frame['transdate']))
    .groupby(['bizym', 'transdate'], as_index=False)
    .agg(qty=('qty', 'sum'), num_hosp=('num_hosp', 'sum'))
    .sort_values(['bizym', 'transdate'])
    .reset_index(drop=True)
)
daily_observed['qty'] = daily_observed['qty'].clip(lower=0)

try:
    from chinese_calendar import is_workday
    WORKDAY_SOURCE = 'chinese_calendar.is_workday'
except ImportError:
    def is_workday(value):
        return value.weekday() < 5
    WORKDAY_SOURCE = 'weekday fallback (install chinese_calendar for production parity)'
    print('WARNING:', WORKDAY_SOURCE)

def complete_month_calendar(frame: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    for bizym, group in frame.groupby('bizym', sort=True):
        period = pd.Period(str(int(bizym)), freq='M')
        calendar = pd.DataFrame({'transdate': pd.date_range(period.start_time, period.end_time, freq='D')})
        completed = calendar.merge(group, on='transdate', how='left')
        completed['bizym'] = int(bizym)
        completed[['qty', 'num_hosp']] = completed[['qty', 'num_hosp']].fillna(0)
        pieces.append(completed)
    return pd.concat(pieces, ignore_index=True).sort_values(['bizym', 'transdate']).reset_index(drop=True)

daily = complete_month_calendar(daily_observed)
daily['is_workday'] = daily['transdate'].map(is_workday).astype(bool)
# 30d 场景的月内滚动预测以法定工作日节奏为主；非工作日不参与状态密度估计。
workday = daily.loc[daily['is_workday']].copy()
print({
    'observed_rows': len(daily_observed),
    'calendar_rows_after_completion': len(daily),
    'workday_rows': len(workday),
    'months': workday['bizym'].nunique(),
    'date_range': f"{workday['transdate'].min().date()} to {workday['transdate'].max().date()}",
    'workday_source': WORKDAY_SOURCE,
    'input_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()[:12],
})
workday.head()

{'observed_rows': 1606, 'calendar_rows_after_completion': 1612, 'workday_rows': 1096, 'months': 53, 'date_range': '2022-01-04 to 2026-05-29', 'workday_source': 'chinese_calendar.is_workday', 'input_sha256': 'fc12013ea2bf'}


,transdate,bizym,qty,num_hosp,is_workday
3,2022-01-04,202201,76524.3317,460.0,True
4,2022-01-05,202201,47902.5417,316.0,True
5,2022-01-06,202201,89947.0000,397.0,True
6,2022-01-07,202201,95147.5417,380.0,True
9,2022-01-10,202201,135945.0000,686.0,True


In [3]:
def add_state_features(frame: pd.DataFrame, vol_window: int, min_periods: int) -> pd.DataFrame:
    """Create within-month features available after each workday closes."""
    out = frame.sort_values(['bizym', 'transdate']).copy()
    out['log_qty'] = np.log1p(out['qty'])
    out['log_growth'] = out.groupby('bizym')['log_qty'].diff()
    out['volatility'] = (
        out.groupby('bizym')['log_growth']
        .transform(lambda values: values.rolling(vol_window, min_periods=min_periods).std(ddof=0))
    )
    # In production, t+1 的特征只能取 t 收盘后已知的状态。
    out['forecast_volatility'] = out.groupby('bizym')['volatility'].shift(1)
    out['direction'] = np.select(
        [out['log_growth'] > 0, out['log_growth'] < 0],
        ['growth', 'decline'],
        default='flat',
    )
    return out

state_daily = add_state_features(workday, VOL_WINDOW, MIN_PERIODS)
state_obs = state_daily.dropna(subset=['log_growth', 'volatility']).copy()
if len(state_obs) < 20:
    raise ValueError(f'Only {len(state_obs)} usable observations; decrease MIN_PERIODS or add history before estimating a density.')

state_obs[['transdate', 'bizym', 'qty', 'log_growth', 'volatility', 'direction']].head()

,transdate,bizym,qty,log_growth,volatility,direction
10,2022-01-11,202201,102974.0000,-0.277772,0.401595,decline
11,2022-01-12,202201,95396.0000,-0.076439,0.370082,decline
12,2022-01-13,202201,81338.7950,-0.159412,0.349437,decline
13,2022-01-14,202201,47623.0000,-0.535299,0.363000,decline
16,2022-01-17,202201,119099.7083,0.916633,0.439869,growth


In [4]:
# KDE is fitted in standardized coordinates so neither axis dominates bandwidth selection.
X = state_obs[['log_growth', 'volatility']].to_numpy().T
feature_mean = X.mean(axis=1, keepdims=True)
feature_std = X.std(axis=1, keepdims=True)
if np.any(feature_std == 0):
    raise ValueError('A state dimension has zero variance; a joint density cannot be estimated.')
X_scaled = (X - feature_mean) / feature_std
kde = gaussian_kde(X_scaled)

growth_grid = np.linspace(*np.quantile(state_obs['log_growth'], [0.005, 0.995]), GRID_SIZE)
vol_grid = np.linspace(max(0, state_obs['volatility'].min()), np.quantile(state_obs['volatility'], 0.995), GRID_SIZE)
growth_mesh, vol_mesh = np.meshgrid(growth_grid, vol_grid)
grid = np.vstack([growth_mesh.ravel(), vol_mesh.ravel()])
grid_scaled = (grid - feature_mean) / feature_std
# Convert the density from z-scored coordinates back to original units.
density = kde(grid_scaled).reshape(growth_mesh.shape) / np.prod(feature_std)

state_summary = state_obs[['log_growth', 'volatility']].describe(percentiles=[0.05, 0.5, 0.95]).round(4)
display(state_summary)
display(state_obs['direction'].value_counts().rename_axis('direction').to_frame('days'))

,log_growth,volatility
count,831.0000,831.0000
mean,-0.0201,0.4896
std,0.5027,0.1851
min,-2.5430,0.1595
5%,-0.5739,0.3018
50%,-0.1378,0.4597
95%,0.9241,0.7565
max,4.0140,1.8899


,days
direction,
decline,545
growth,286


In [5]:
surface = go.Surface(
    x=growth_mesh,
    y=vol_mesh,
    z=density,
    colorscale='Viridis',
    colorbar=dict(title='density'),
    contours=dict(z=dict(show=True, usecolormap=True, project=dict(z=True))),
    hovertemplate='log growth: %{x:.4f}<br>volatility: %{y:.4f}<br>density: %{z:.4f}<extra></extra>',
)
fig = go.Figure(surface)
fig.update_layout(
    title=f'30d-jenny joint density: log growth × {VOL_WINDOW}-workday volatility',
    template='plotly_white',
    width=950,
    height=760,
    scene=dict(
        xaxis_title='signed log growth: log1p(qty_t) - log1p(qty_t-1)',
        yaxis_title=f'{VOL_WINDOW}-workday volatility of log growth',
        zaxis_title='joint probability density',
        camera=dict(eye=dict(x=1.5, y=-1.6, z=1.05)),
    ),
)
fig.show()

## GMM-3 regime labeling

GMM 在标准化后的 `[log_growth, volatility]` 上拟合，避免增长率的尺度主导协方差估计。`regime` 是最大后验概率对应的统计成分；它是探索性标签，不是已经验证可预测的 HMM 隐状态。

In [6]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

N_REGIMES = 3
RANDOM_STATE = 42

gmm_feature_cols = ['log_growth', 'volatility']
scaler_gmm = StandardScaler()
X_gmm = scaler_gmm.fit_transform(state_obs[gmm_feature_cols])
gmm3 = GaussianMixture(
    n_components=N_REGIMES,
    covariance_type='full',
    n_init=50,
    reg_covar=1e-5,
    random_state=RANDOM_STATE,
).fit(X_gmm)

state_obs = state_obs.copy()
state_obs['gmm_component'] = gmm3.predict(X_gmm)
posterior = gmm3.predict_proba(X_gmm)
state_obs['regime_probability'] = posterior.max(axis=1)
state_obs['regime_entropy'] = -(posterior * np.log(posterior.clip(1e-12))).sum(axis=1)

# Component numbers are arbitrary. Reorder by mean volatility, then mean growth for stable labels and colors.
profile = (
    state_obs.groupby('gmm_component')
    .agg(
        days=('gmm_component', 'size'),
        mean_log_growth=('log_growth', 'mean'),
        mean_volatility=('volatility', 'mean'),
        mean_posterior=('regime_probability', 'mean'),
    )
    .sort_values(['mean_volatility', 'mean_log_growth'])
)
component_to_regime = {component: f'Regime {rank + 1}' for rank, component in enumerate(profile.index)}
state_obs['regime'] = state_obs['gmm_component'].map(component_to_regime)
profile['regime'] = profile.index.map(component_to_regime)
profile['share'] = profile['days'] / len(state_obs)
profile = profile.set_index('regime')[['days', 'share', 'mean_log_growth', 'mean_volatility', 'mean_posterior']].round(3)

print(f'GMM-{N_REGIMES}: BIC={gmm3.bic(X_gmm):.1f}; AIC={gmm3.aic(X_gmm):.1f}; observations={len(state_obs)}')
display(profile)

GMM-3: BIC=3574.0; AIC=3493.8; observations=831


,days,share,mean_log_growth,mean_volatility,mean_posterior
regime,,,,,
Regime 1,650,0.782,-0.200,0.446,0.980
Regime 2,133,0.160,0.841,0.511,0.967
Regime 3,48,0.058,0.036,1.016,0.943


### Regime-colored time series

灰线保留连续的数值轨迹；彩色小点是 GMM-3 的最大后验 regime。点的 hover 信息包含该标签的后验概率，低概率点位于成分边界，不能过度解读。

In [7]:
REGIME_COLORS = {
    'Regime 1': '#4C78A8',
    'Regime 2': '#F58518',
    'Regime 3': '#54A24B',
}

def plot_regime_series(value_col: str, title: str, yaxis_title: str, add_zero_line: bool = False) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=state_obs['transdate'], y=state_obs[value_col],
        mode='lines', name='observed value',
        line=dict(color='#B8C2CC', width=1), hoverinfo='skip',
    ))
    for regime in sorted(state_obs['regime'].unique()):
        subset = state_obs.loc[state_obs['regime'] == regime]
        fig.add_trace(go.Scatter(
            x=subset['transdate'], y=subset[value_col],
            mode='markers', name=regime,
            marker=dict(color=REGIME_COLORS[regime], size=6, opacity=0.85),
            customdata=np.column_stack([subset['regime_probability'], subset['regime_entropy'], subset['qty']]),
            hovertemplate=(
                'date: %{x|%Y-%m-%d}<br>' + yaxis_title + ': %{y:.4f}<br>'
                'posterior: %{customdata[0]:.3f}<br>entropy: %{customdata[1]:.3f}<br>qty: %{customdata[2]:,.0f}<extra>' + regime + '</extra>'
            ),
        ))
    if add_zero_line:
        fig.add_hline(y=0, line_dash='dash', line_color='#555555', line_width=1)
    fig.update_layout(
        title=title, template='plotly_white', width=1100, height=430,
        xaxis_title='date', yaxis_title=yaxis_title,
        legend_title='GMM-3 regime', hovermode='closest',
    )
    return fig

fig_volatility = plot_regime_series(
    'volatility',
    f'30d-jenny: {VOL_WINDOW}-workday volatility over time, colored by GMM-3 regime',
    'volatility',
)
fig_volatility.show()

In [8]:
fig_log_growth = plot_regime_series(
    'log_growth',
    '30d-jenny: signed log growth over time, colored by GMM-3 regime',
    'signed log growth',
    add_zero_line=True,
)
fig_log_growth.show()

## Interpretation and next step

- 密度峰表示历史上最常见的月内变化状态；峰之间若有清晰分隔，才支持继续尝试 2–3 状态的 GMM/HMM。
- 单一连续峰不代表 HMM 无效，但意味着状态划分未必优于 EWMA / Kalman / 变点模型。
- 下一步的 HMM 观测向量建议从 `[log_growth, volatility, completion_residual]` 开始；`completion_residual` 必须仅由截止当天的 MTD 与历史可见基准构造。
- 评估必须按月滚动 OOS：状态预测是否改善下一工作日/最终月总量的 WAPE、Bias 和预测区间覆盖率。